# Lecture 2 — Class Exercise
# Bar Charts: World Happiness Report 2023

---

> **Your task:** Create **2 polished bar charts** using the World Happiness Report dataset.  
> **Push to:** `week02/lecture02_exercise.ipynb` in **your own GitHub repo** before the end of class.

---

### Rules to keep in mind 💡
1. Every bar chart **must have a zero baseline** — no exceptions
2. Every chart **must have an insight title**, not a topic title
3. Aim for **professional quality** — clean background, readable font, no clutter
4. Horizontal bars for long category names

---


## Setup — Run this cell first


In [2]:
import pandas as pd
import numpy as np

# World Happiness Report 2023 — representative data
# Source: https://www.kaggle.com/datasets/ajaypalsinghlo/world-happiness-report-2023

df = pd.read_csv('../data/world_happiness_2023.csv')
df.columns = ['Country','Region','Happiness_Score','GDP','Social_Support',
              'Life_Expectancy','Freedom','Generosity','Corruption']


print(f"Dataset: {len(df)} countries, {len(df.columns)} columns")
df.head()


Dataset: 63 countries, 9 columns


,Country,Region,Happiness_Score,GDP,Social_Support,Life_Expectancy,Freedom,Generosity,Corruption
0,Finland,Western Europe,7.804,10.775,0.954,71.9,0.949,0.142,0.179
1,Denmark,Western Europe,7.586,10.933,0.954,72.7,0.931,0.168,0.234
2,Iceland,Western Europe,7.525,10.878,0.983,72.5,0.961,0.260,0.150
3,Israel,Middle East and North Africa,7.473,10.527,0.916,72.4,0.903,0.149,0.826
4,Netherlands,Western Europe,7.464,11.015,0.939,72.4,0.879,0.240,0.296


In [3]:
import plotly.express as px
import plotly.graph_objects as go

# Explore the dataset before you start
print("Regions in dataset:")
print(df['Region'].value_counts())
print("\nScore range:", df['Happiness_Score'].min(), "–", df['Happiness_Score'].max())
print("\nBottom 10 countries:")
print(df.nsmallest(10, 'Happiness_Score')[['Country','Region','Happiness_Score']])


Regions in dataset:
Region
Western Europe                  15
Latin America and Caribbean     13
Central and Eastern Europe       7
Sub-Saharan Africa               7
Middle East and North Africa     6
North America and ANZ            4
Southeast Asia                   4
South Asia                       4
East Asia                        3
Name: count, dtype: int64

Score range: 1.859 – 7.804

Bottom 10 countries:
        Country                        Region  Happiness_Score
60  Afghanistan                    South Asia            1.859
61      Lebanon  Middle East and North Africa            2.392
62     Zimbabwe            Sub-Saharan Africa            2.995
52     Ethiopia            Sub-Saharan Africa            3.564
53     Tanzania            Sub-Saharan Africa            3.698
48   Bangladesh                    South Asia            3.892
47        India                    South Asia            4.036
50        Kenya            Sub-Saharan Africa            4.112
54       Uganda

---
## Task 1 — Regional Comparison Bar Chart

**What to build:** A horizontal bar chart showing the **average happiness score by region**, sorted from highest to lowest.

**Requirements:**
- Horizontal orientation (category names are long)
- Sorted by score, descending (so the happiest region is at the top)
- Zero baseline on x-axis
- At least one design choice that goes beyond the Plotly default (colour, annotation, labels, etc.)
- An insight title that answers: *which region stands out and why does it matter?*

**Hint:** Use `df.groupby('Region')['Happiness_Score'].mean()` to compute the averages.


In [ ]:
# Task 1: Regional comparison bar chart
# -------------------------------------

# Step 1: Compute average happiness score by region
region_avg = (df.groupby('Region')['Happiness_Score']
              .mean()
              .reset_index()
              .sort_values('Happiness_Score'))  # sort for horizontal bar

print(region_avg)

# Step 2: Build your chart
top_region = region_avg.iloc[-1]
low_region = region_avg.iloc[0]
ratio = top_region['Happiness_Score'] / low_region['Happiness_Score']

# Highlight the top and bottom regions, grey out the rest
colors = ['#C0392B' if r == low_region['Region']
          else '#1F77B4' if r == top_region['Region']
          else '#BFC5CC' for r in region_avg['Region']]

fig1 = go.Figure(go.Bar(
    x=region_avg['Happiness_Score'],
    y=region_avg['Region'],
    orientation='h',
    marker_color=colors,
    text=region_avg['Happiness_Score'].round(2),
    textposition='outside',
    hovertemplate='%{y}: %{x:.2f}<extra></extra>'
))

fig1.update_layout(
    title=dict(
        text=(f"<b>{top_region['Region']} is {ratio:.1f}x happier than {low_region['Region']}</b>"
              "<br><sup>Average happiness score by region, World Happiness Report 2023 (0–10 scale)</sup>"),
        x=0, xanchor='left'
    ),
    template='simple_white',
    font=dict(family='Arial', size=13),
    xaxis=dict(title='Average happiness score', range=[0, 10], rangemode='tozero'),
    yaxis=dict(title=None),
    height=480, margin=dict(l=20, r=40, t=90, b=50),
    showlegend=False
)
fig1.show()


---
## Task 2 — Bottom vs. Top: A Contrast Story

**What to build:** A bar chart that highlights the **gap between the happiest and least happy countries**, focusing on a specific insight.

**Requirements:**
- Show the **top 8 AND bottom 8 countries** together (16 bars total)
- Use **colour** to distinguish the two groups (not Plotly's default rainbow)
- Add a **visual separator or annotation** that emphasises the gap
- Insight title that tells the story of the gap

**Hint:** Use `pd.concat([df.nlargest(8,'Happiness_Score'), df.nsmallest(8,'Happiness_Score')])` to get both groups.

**Stretch goal:** Add a vertical reference line showing the global average.


In [ ]:
# Task 2: Top 8 vs. Bottom 8 contrast
# ------------------------------------

# Step 1: Get top and bottom countries
top8 = df.nlargest(8, 'Happiness_Score').copy()
top8['Group'] = 'Top 8'
bottom8 = df.nsmallest(8, 'Happiness_Score').copy()
bottom8['Group'] = 'Bottom 8'

combined = pd.concat([bottom8, top8]).sort_values('Happiness_Score')
global_avg = df['Happiness_Score'].mean()
print(f"Global average: {global_avg:.2f}")

# Step 2: Build your chart
happiest = top8.iloc[0]
unhappiest = bottom8.iloc[0]
gap_ratio = happiest['Happiness_Score'] / unhappiest['Happiness_Score']

fig2 = px.bar(
    combined, x='Happiness_Score', y='Country', orientation='h',
    color='Group',
    color_discrete_map={'Top 8': '#1F77B4', 'Bottom 8': '#C0392B'},
    text=combined['Happiness_Score'].round(2),
    hover_data={'Region': True, 'Group': False}
)
fig2.update_traces(textposition='outside')

# Global average reference line (stretch goal)
fig2.add_vline(x=global_avg, line_dash='dash', line_color='#555',
               annotation_text=f'Global avg {global_avg:.2f}',
               annotation_position='top')

# Separator between the two groups
fig2.add_hline(y=7.5, line_color='#999', line_width=1)
fig2.add_annotation(
    x=9.2, y=7.5, yshift=0,
    text=f"<b>Gap: {happiest['Happiness_Score'] - unhappiest['Happiness_Score']:.1f} points</b>",
    showarrow=False, bgcolor='white', font=dict(size=12, color='#333')
)

fig2.update_layout(
    title=dict(
        text=(f"<b>{happiest['Country']} scores {gap_ratio:.1f}x higher than {unhappiest['Country']}</b>"
              "<br><sup>Top 8 vs. bottom 8 countries by happiness score, 2023 — "
              "the bottom group sits well below the global average</sup>"),
        x=0, xanchor='left'
    ),
    template='simple_white',
    font=dict(family='Arial', size=13),
    xaxis=dict(title='Happiness score (0–10)', range=[0, 10], rangemode='tozero'),
    yaxis=dict(title=None, categoryorder='array', categoryarray=combined['Country'].tolist()),
    legend=dict(title=None, orientation='h', y=1.02, x=1, xanchor='right', yanchor='bottom'),
    height=620, margin=dict(l=20, r=40, t=110, b=50)
)
fig2.show()


---
## Done? Stretch Goal

If you finish both tasks with time to spare, try this:

**Task 3 (stretch):** Build a **grouped bar chart** comparing 2 sub-factors (e.g. `GDP_per_capita` and `Freedom`) across the 5 most populated regions. Use colour meaningfully and write an insight title.

Regions to include: `'Western Europe'`, `'Latin America'`, `'East Asia'`, `'Sub-Saharan Africa'`, `'South Asia'`


In [ ]:
# Stretch goal — grouped bar chart
# GDP (log scale) and Freedom (0–1) have different units, so rescale each to 0–1
# (min–max across all countries) before comparing them side by side.
regions = ['Western Europe', 'Latin America and Caribbean', 'East Asia',
           'Sub-Saharan Africa', 'South Asia']

scaled = df.copy()
for col in ['GDP', 'Freedom']:
    scaled[col] = (df[col] - df[col].min()) / (df[col].max() - df[col].min())

factors = (scaled[scaled['Region'].isin(regions)]
           .groupby('Region')[['GDP', 'Freedom']].mean()
           .reindex(regions)
           .sort_values('GDP', ascending=False)
           .reset_index()
           .melt(id_vars='Region', var_name='Factor', value_name='Score'))

wide = factors.pivot(index='Region', columns='Factor', values='Score')
wide['Gap'] = wide['Freedom'] - wide['GDP']
lead = wide['Gap'].idxmax()   # region where freedom most outpaces wealth

fig3 = px.bar(
    factors, x='Region', y='Score', color='Factor', barmode='group',
    color_discrete_map={'GDP': '#1F77B4', 'Freedom': '#F39C12'},
    text=factors['Score'].round(2)
)
fig3.update_traces(textposition='outside')
fig3.update_layout(
    title=dict(
        text=(f"<b>In {lead}, freedom outpaces wealth the most "
              f"({wide.loc[lead, 'Freedom']:.2f} vs {wide.loc[lead, 'GDP']:.2f})</b>"
              "<br><sup>Average GDP per capita (log) and freedom, each rescaled 0–1 across all countries</sup>"),
        x=0, xanchor='left'
    ),
    template='simple_white',
    font=dict(family='Arial', size=13),
    yaxis=dict(title='Relative score (0 = lowest country, 1 = highest)', range=[0, 1.1], rangemode='tozero'),
    xaxis=dict(title=None),
    legend=dict(title=None, orientation='h', y=1.02, x=1, xanchor='right', yanchor='bottom'),
    height=500, margin=dict(l=20, r=20, t=110, b=50)
)
fig3.show()
